# 🧪 Phase 4C: THRS v3 Ingredient / Formulation Signal Component Design (Policy Lock)
### *Qualitative Formulation Signals • Policy-Based Weights • Double-Counting Safeguards • 5-Product Benchmark*

---

> **System Boundaries & Governance:**
> * Phase 4B (Mathematical Nutrition Model) is **FROZEN** and validated.
> * THRS v2.0 remains **FROZEN and untouched** (`health_scores.json`, `multinational_brand_cleaned.csv`, `recommendations.json`, `app.py`).
> * **NO JECFA retrieval** is performed in this phase.
> * This notebook audits existing ingredient signals in the repository, prevents double-counting with $P_{\text{nutrition}}$, and locks a small, bounded $P_{\text{ingredient}}$ component.
> * **NO final THRS v3 scores** are combined or saved to production in this phase.


## 📚 Part 1: Conceptual Foundations of the Ingredient / Formulation Signal Component

---

### 1. What $P_{\text{ingredient}}$ Represents
While the Nutrition Component ($P_{\text{nutrition}}$) evaluates **quantitative macronutrient density** (grams of sugar, saturated fat, and milligrams of sodium), the Ingredient Component ($P_{\text{ingredient}}$) evaluates **qualitative formulation signals**:
* Synthetic chemical colorants (e.g. Azo dyes and non-azo synthetic dyes)
* Industrial chemical preservatives (e.g. Benzoates, Sorbates, Sulfites)
* Hyper-palatability flavor enhancers (e.g. MSG, Ribonucleotides)
* Artificial non-nutritive high-intensity sweeteners (e.g. Sucralose, Aspartame)
* Industrial synthetic fat replacers / high-intensity emulsifiers (e.g. PGPR, DATEM)
* Explicitly declared refined palm oils (e.g. Refined Palmolein)

These formulation characteristics are **independent of caloric density** and are not captured on the standard macronutrient facts panel.

---

### 2. Crucial Principle: Model Policy Weights vs. Clinical Toxicity
* **AntiGravity Model Policy Weights ($-3, -4, -5, -6, -8$):** These numerical values are deliberate **engineering design weights** within the AntiGravity scoring system. They are **NOT** statutory penalties mandated by law, nor are they clinical toxicity measurements.
* **Presence $\neq$ Automatic Harm:** The mere qualitative presence of an approved food additive on a label does not prove acute toxicity, clinical disease causation, or unsafe consumer exposure. It functions strictly as an **informational formulation signal** indicating the level of industrial ultra-processing and chemical formulation complexity.

---

### 3. Why We Must NOT Penalize Every E-Number Indiscriminately ("Chemophobia")
An **INS (International Numbering System) or E-number** is simply an international standardized identifier for food substances.
* Many E-numbers are completely benign, essential nutrients, or natural plant extracts:
  * **INS 300:** Ascorbic Acid (Pure Vitamin C)
  * **INS 330:** Citric Acid (Naturally occurring citrus acid)
  * **INS 322:** Soya/Sunflower Lecithin (Natural phospholipid from plant seeds)
  * **INS 414:** Gum Arabic (Natural acacia tree sap)
  * **INS 500(ii):** Sodium Bicarbonate (Baking soda)
* **The Rule:** Penalizing every E-number creates unscientific "chemophobia." Additives must be categorized into functional classes: `NEUTRAL` ($0\text{ penalty}$) vs `FORMULATION_SIGNAL / WATCH` vs `EVIDENCE_SUPPORTED_SIGNAL`.

---

### 4. Regulatory Divergence is Context, Not Automatic Proof of Harm
* When a food additive is subject to special labeling in one jurisdiction (e.g. EU requiring warning notices on Tartrazine INS 102) while permitted as standard in another (e.g. India or US FDA), it reflects differing **precautionary risk policies, statutory dietary exposure models, or historical regulatory standards**.
* Regulatory divergence serves as **investigative context and a transparency flag**, not an automatic universal declaration of acute biological harm.

---

### 5. The "Innocent Until Proven Guilty" Rule for Unknown/Unverified Ingredients
* If an ingredient string is unrecognized due to OCR typos or represents a novel botanical extract without established safety concerns, it must **never silently receive a deduction**.
* It must be categorized as `UNKNOWN / UNVERIFIED` ($0\text{ numeric penalty}$), generating an informational flag for human review.

---

### 6. Why $P_{\text{ingredient}}$ Must Have a Hard Component Cap
* If a complex snack contains multiple minor processing aids (e.g. an antioxidant, an emulsifier, a raising agent, and palm oil), uncapped stacking could artificially zero out a product's health score.
* A hard component ceiling ($P_{\text{ingredient}} \le \mathbf{25.0\text{ points}}$) ensures that formulation deductions remain bounded, fair, and interpretable.


## 📊 Part 2: Audit of Existing Project Ingredient Signals

We inspect the actual ingredient-level fields present in:
1. `data/multinational_brand_cleaned.csv` (`Ingredients` raw text)
2. `data/llm_ingredient_intelligence.json` (`decoded_e_numbers` and `key_difference`)


In [1]:
import json
import re
import pandas as pd
from collections import Counter
from pathlib import Path

data_dir = Path('data')

# Load datasets
with open(data_dir / 'llm_ingredient_intelligence.json', 'r', encoding='utf-8') as f:
    llm_data = json.load(f)
df_clean = pd.read_csv(data_dir / 'multinational_brand_cleaned.csv')

print(f"Total Products with Extracted Ingredient Intelligence: {len(llm_data)}")
print(f"Total Cleaned CSV Production Products: {len(df_clean)}")

# 1. Frequency of E/INS Codes in Project
e_counter = Counter()
e_to_names = {}
for r in llm_data:
    dec = r.get('decoded_e_numbers', {})
    if isinstance(dec, dict):
        for code, name in dec.items():
            clean_c = str(code).upper().replace('E-', '').replace('E', '').replace('INS', '').replace(' ', '').strip()
            e_counter[clean_c] += 1
            if clean_c not in e_to_names:
                e_to_names[clean_c] = set()
            e_to_names[clean_c].add(str(name).strip())

print(f"\nUnique E/INS Additive Codes Identified: {len(e_counter)}")
print("\nTop 15 Most Frequent E/INS Additives in Dataset:")
top_e_records = []
for c, cnt in e_counter.most_common(15):
    top_e_records.append({
        'INS Code': f"INS {c}",
        'Product Count': cnt,
        'Frequency (%)': f"{cnt/len(llm_data)*100:.1f}%",
        'Example Name(s)': ', '.join(list(e_to_names[c])[:2])
    })
df_top_e = pd.DataFrame(top_e_records)
print(df_top_e.to_string(index=False))

# 2. Frequency of Qualitative Functional Classes in Cleaned CSV Text
def clean_recipe_text(text):
    if pd.isna(text): return ''
    s = str(text)
    # Strip allergen / cross-contamination warnings before keyword scanning
    return re.split(r'allergen\s+information|may\s+contain', s, flags=re.IGNORECASE)[0].lower()

ing_text_cleaned = df_clean['Ingredients'].apply(clean_recipe_text)

functional_signals = {
    'Explicit Refined Palm Oil / Palmolein': ing_text_cleaned.str.contains('palm', regex=False),
    'Chemical Preservatives (Benzoates/Sorbates/Sulfites)': ing_text_cleaned.str.contains('benzoate', regex=False) | ing_text_cleaned.str.contains('sorbate', regex=False) | ing_text_cleaned.str.contains('211', regex=False) | ing_text_cleaned.str.contains('202', regex=False),
    'Synthetic Colors (Azo Dyes & Brilliant Blue 133)': ing_text_cleaned.str.contains('tartrazine', regex=False) | ing_text_cleaned.str.contains('sunset yellow', regex=False) | ing_text_cleaned.str.contains('brilliant blue', regex=False) | ing_text_cleaned.str.contains('102', regex=False) | ing_text_cleaned.str.contains('110', regex=False) | ing_text_cleaned.str.contains('124', regex=False) | ing_text_cleaned.str.contains('129', regex=False) | ing_text_cleaned.str.contains('133', regex=False),
    'Flavor Enhancers (MSG/Inosinate/Guanylate/Ribonucleotides)': ing_text_cleaned.str.contains('621', regex=False) | ing_text_cleaned.str.contains('627', regex=False) | ing_text_cleaned.str.contains('631', regex=False) | ing_text_cleaned.str.contains('635', regex=False) | ing_text_cleaned.str.contains('glutamate', regex=False) | ing_text_cleaned.str.contains('inosinate', regex=False) | ing_text_cleaned.str.contains('guanylate', regex=False),
    'Industrial Emulsifiers (PGPR 476, YN 442, DATEM 472e, 471)': ing_text_cleaned.str.contains('476', regex=False) | ing_text_cleaned.str.contains('442', regex=False) | ing_text_cleaned.str.contains('471', regex=False) | ing_text_cleaned.str.contains('polyricinoleate', regex=False),
    'Benign / Natural Hydrocolloids (Lecithin 322, Gum Arabic 414)': ing_text_cleaned.str.contains('322', regex=False) | ing_text_cleaned.str.contains('414', regex=False) | ing_text_cleaned.str.contains('lecithin', regex=False) | ing_text_cleaned.str.contains('acacia', regex=False),
    'Benign Acidity Regulators (Citric 330, Malic 296, Tartaric 334)': ing_text_cleaned.str.contains('330', regex=False) | ing_text_cleaned.str.contains('296', regex=False) | ing_text_cleaned.str.contains('334', regex=False) | ing_text_cleaned.str.contains('citric acid', regex=False),
    'Artificial / High-Intensity Sweeteners (Sucralose, Aspartame, Ace-K)': ing_text_cleaned.str.contains('sucralose', regex=False) | ing_text_cleaned.str.contains('aspartame', regex=False) | ing_text_cleaned.str.contains('acesulfame', regex=False) | ing_text_cleaned.str.contains('950', regex=False) | ing_text_cleaned.str.contains('951', regex=False) | ing_text_cleaned.str.contains('955', regex=False)
}

print("\n=== FREQUENCY OF REUSABLE FUNCTIONAL CLASSES ACROSS INVENTORY ===")
func_rows = []
for sname, mask in functional_signals.items():
    func_rows.append({
        'Functional Ingredient Class': sname,
        'Affected Products': mask.sum(),
        'Prevalence (%)': f"{mask.sum()/len(df_clean)*100:.1f}%"
    })
df_func = pd.DataFrame(func_rows)
print(df_func.to_string(index=False))


Total Products with Extracted Ingredient Intelligence: 151
Total Cleaned CSV Production Products: 172

Unique E/INS Additive Codes Identified: 255

Top 15 Most Frequent E/INS Additives in Dataset:
   INS Code  Product Count Frequency (%)                                                                Example Name(s)
    INS 330             38         25.2%                                                       Citric Acid, Citric acid
    INS 476             37         24.5%                     Polyglyceryl polyricinoleate, Polyglycerol polyricinoleate
    INS 322             33         21.9%                                                        Lecithin, Soya Lecithin
    INS 442             30         19.9%                                     Ammonium phosphatides, Potassium phosphate
    INS 471             27         17.9% Mono- and di-glycerides of fatty acids, Mono- and Di-Glycerides of Fatty Acids
    INS 414             21         13.9%                                           

## 🛡️ Part 3 & 4: Double-Counting Audit & Policy Matrix

Before assigning any penalty, we must verify whether the underlying health concern is **already penalized in the Nutrition Component ($P_{\text{nutrition}}$)**:

---

### Double-Counting Audit Matrix:

| Candidate Signal | Measured Nutrient Overlap | Double-Counting Risk | Proposed Policy Handling |
|---|---|---|---|
| **Cane Sugar / Corn Syrup / Invert Sugar** | Overlaps $100\%$ with `Sugar_g` | **HIGH** | ❌ **EXCLUDE from $P_{\text{ingredient}}$**. Sugar is already penalized quantitatively up to $-20.0\text{ pts}$ in $P_{\text{nutrition}}$. Penalizing "Sugar" text again double-penalizes the exact same grams of sucrose. |
| **Table Salt / Sodium Chloride** | Overlaps $100\%$ with `Sodium_mg` | **HIGH** | ❌ **EXCLUDE from $P_{\text{ingredient}}$**. Sodium is already penalized up to $-15.0\text{ pts}$ in $P_{\text{nutrition}}$. |
| **Refined Palm Oil / Palmolein** | Overlaps partially with `Saturated_Fat_g` | **MEDIUM** | ⚠️ **INCLUDE as `FORMULATION_SIGNAL / WATCH` ($-3.0\text{ pts}$)**. Retained as a separate industrial formulation/processing signal rather than a second measurement of saturated fat. |
| **Artificial Sweeteners (Sucralose, Aspartame, Ace-K)** | **Zero overlap with `Sugar_g`** (`Sugar_g = 0` in diet foods) | **NONE** | ✅ **INCLUDE as `EVIDENCE_SUPPORTED_SIGNAL` ($-6.0\text{ pts}$)**. Non-nutritive sweeteners escape $P_{\text{nutrition}}$ entirely. Evaluating them here captures sweet-taste habituation and non-nutritive formulation. |
| **Synthetic Food Colors (Azo Dyes & Triarylmethane INS 133)** | **Zero overlap with nutrition** | **NONE** | ✅ **INCLUDE as `EVIDENCE_SUPPORTED_SIGNAL` ($-8.0\text{ pts}$)**. Synthetic colors subject to cross-border statutory warnings. |
| **Chemical Preservatives (Sodium Benzoate INS 211, Sorbates)** | **Zero overlap with nutrition** | **NONE** | ✅ **INCLUDE as `FORMULATION_SIGNAL / WATCH` ($-5.0\text{ pts}$)**. Industrial chemical preservatives. |
| **Flavor Enhancers (MSG INS 621, Ribonucleotides INS 635)** | **Negligible overlap with sodium** | **NONE** | ✅ **INCLUDE as `FORMULATION_SIGNAL / WATCH` ($-4.0\text{ pts}$)**. Industrial flavor enhancers. |
| **Industrial Emulsifiers (PGPR INS 476, YN INS 442, DATEM INS 472e)**| **Zero overlap with nutrition** | **NONE** | ✅ **INCLUDE as `FORMULATION_SIGNAL / WATCH` ($-4.0\text{ pts}$)**. Synthetic fat replacers / high-intensity emulsifiers. |
| **Natural / Benign Additives (Citric Acid, Lecithin, Gum Arabic)**| **Zero overlap** | **NONE** | 🛡️ **`NEUTRAL` ($0.0\text{ pts}$)**. Naturally occurring organic acids, vitamins, and plant hydrocolloids. |


## 📐 Part 5 & 6: Bounded Policy & Numerical Formulation

### The 4-Tier Qualitative Policy:
1. **`EVIDENCE_SUPPORTED_SIGNAL` ($-6.0\text{ to } -8.0\text{ pts}$ per distinct class):**
   * **Synthetic Colors:**
     * *Azo Dyes:* Tartrazine (INS 102), Sunset Yellow (INS 110), Ponceau 4R (INS 124), Allura Red (INS 129) $\rightarrow \mathbf{-8.0\text{ pts}}$
     * *Triarylmethane Dye:* Brilliant Blue FCF (INS 133) $\rightarrow \mathbf{-8.0\text{ pts}}$
   * **Artificial Sweeteners:** Sucralose (INS 955), Aspartame (INS 951), Acesulfame-K (INS 950), Saccharin (INS 954) $\rightarrow \mathbf{-6.0\text{ pts}}$
2. **`FORMULATION_SIGNAL / WATCH` ($-3.0\text{ to } -5.0\text{ pts}$ per distinct class):**
   * **Chemical Preservatives:** Sodium Benzoate (INS 211), Potassium Sorbate (INS 202), Sulfites in recipe (INS 220–228) $\rightarrow \mathbf{-5.0\text{ pts}}$
   * **Flavor Enhancers:** MSG (INS 621), Disodium Inosinate/Guanylate/Ribonucleotides (INS 627, 631, 635) $\rightarrow \mathbf{-4.0\text{ pts}}$
   * **Industrial Emulsifiers:** PGPR (INS 476), Ammonium Phosphatides (INS 442), DATEM (INS 472e), Mono/Diglycerides (INS 471) $\rightarrow \mathbf{-4.0\text{ pts}}$
   * **Explicit Refined Palm Oil / Palmolein:** $\rightarrow \mathbf{-3.0\text{ pts}}$
3. **`NEUTRAL` ($0.0\text{ pts}$ Deduction):**
   * Organic acids (Citric 330, Malic 296, Tartaric 334, Ascorbic 300, Sodium Citrate 331, Bicarbonates 500ii, 503ii).
   * Natural plant gums & phospholipids (Lecithin 322, Gum Arabic 414, Guar Gum 412, Pectin 440).
4. **`UNKNOWN / UNVERIFIED` ($0.0\text{ pts}$ Deduction — Informational Flag Only):**
   * Generic unspecified vegetable oil $\rightarrow$ `GENERIC_VEGETABLE_OIL_FLAG` ($0.0\text{ pts}$)
   * Unrecognized chemical strings or novel botanicals $\rightarrow$ `UNVERIFIED_INGREDIENT_FLAG` ($0.0\text{ pts}$)

---

### Component Stacking & Capping:
* Deductions are triggered **once per functional category** (e.g. a product with two flavor enhancers like INS 627 + INS 631 triggers the flavor enhancer penalty once, $-4.0\text{ pts}$).
* **Hard Component Cap:**
$$P_{\text{ingredient}} = \min\left( \sum P_{\text{signal\_class}}, \; \mathbf{25.0\text{ points}} \right)$$


In [2]:
def detect_ingredient_penalties(ingredients_text, decoded_e_dict=None):
    raw_s = str(ingredients_text) if pd.notna(ingredients_text) else ''
    
    # 1. Clean recipe text: strip allergen / cross-contamination text
    recipe_text = re.split(r'allergen\s+information|may\s+contain', raw_s, flags=re.IGNORECASE)[0].lower()
    
    # 2. Extract E-codes strictly present in recipe text or explicit dictionary
    e_codes = set()
    if isinstance(decoded_e_dict, dict):
        for k in decoded_e_dict.keys():
            clean_k = str(k).upper().replace('E-', '').replace('E', '').replace('INS', '').replace(' ', '').strip()
            if clean_k in recipe_text or f"ins {clean_k.lower()}" in recipe_text or f"({clean_k.lower()})" in recipe_text:
                e_codes.add(clean_k)
            elif not ingredients_text:
                e_codes.add(clean_k)
                
    found_ins = re.findall(r'ins\s*(\d+[a-z]*)', recipe_text)
    for c in found_ins:
        e_codes.add(c.upper().strip())
        
    detected_classes = []
    info_flags = []
    total_penalty = 0.0
    
    # 1. Synthetic Colors (Azo + Triarylmethane) (-8.0 pts)
    azo_codes = {'102', '110', '124', '129'}
    triaryl_codes = {'133'}
    azo_keywords = ['tartrazine', 'sunset yellow', 'allura red', 'ponceau']
    triaryl_keywords = ['brilliant blue']
    
    has_azo = any(c in e_codes for c in azo_codes) or any(k in recipe_text for k in azo_keywords)
    has_triaryl = any(c in e_codes for c in triaryl_codes) or any(k in recipe_text for k in triaryl_keywords)
    
    if has_azo or has_triaryl:
        sub_type = 'Azo Dye' if has_azo else 'Triarylmethane Dye (INS 133)'
        detected_classes.append(('SYNTHETIC_COLORS', 8.0, 'EVIDENCE_SUPPORTED_SIGNAL', f'Synthetic Color ({sub_type})'))
        total_penalty += 8.0
        
    # 2. Artificial Sweeteners (-6.0 pts)
    sweeteners_codes = {'950', '951', '954', '955'}
    sweeteners_keywords = ['sucralose', 'aspartame', 'acesulfame', 'saccharin']
    if any(c in e_codes for c in sweeteners_codes) or any(k in recipe_text for k in sweeteners_keywords):
        detected_classes.append(('ARTIFICIAL_SWEETENERS', 6.0, 'EVIDENCE_SUPPORTED_SIGNAL', 'Artificial High-Intensity Sweetener'))
        total_penalty += 6.0
        
    # 3. Chemical Preservatives (-5.0 pts)
    preservatives_codes = {'211', '202', '220', '221', '222', '223', '224', '228'}
    preservatives_keywords = ['sodium benzoate', 'potassium sorbate', 'preservative (211)', 'preservative (202)', 'benzoate', 'sorbate', 'metabisulphite', 'metabisulfite']
    if any(c in e_codes for c in preservatives_codes) or any(k in recipe_text for k in preservatives_keywords):
        detected_classes.append(('CHEMICAL_PRESERVATIVES', 5.0, 'FORMULATION_SIGNAL / WATCH', 'Industrial Chemical Preservative'))
        total_penalty += 5.0
        
    # 4. Flavor Enhancers (-4.0 pts)
    flavor_codes = {'621', '627', '631', '635'}
    flavor_keywords = ['glutamate', 'inosinate', 'guanylate', 'ribonucleotide', 'msg', 'flavour enhancer (635)', 'flavor enhancer']
    if any(c in e_codes for c in flavor_codes) or any(k in recipe_text for k in flavor_keywords):
        detected_classes.append(('FLAVOR_ENHANCERS', 4.0, 'FORMULATION_SIGNAL / WATCH', 'Industrial Flavor Enhancer'))
        total_penalty += 4.0
        
    # 5. Industrial Synthetic Emulsifiers (-4.0 pts)
    emulsifiers_codes = {'476', '442', '471', '472E', '472'}
    emulsifiers_keywords = ['polyricinoleate', 'pgpr', 'ammonium phosphatide', 'mono- and di-glycerides', 'mono- and diglycerides', 'datem', 'ins 471', 'ins 476']
    if any(c in e_codes for c in emulsifiers_codes) or any(k in recipe_text for k in emulsifiers_keywords):
        detected_classes.append(('INDUSTRIAL_EMULSIFIERS', 4.0, 'FORMULATION_SIGNAL / WATCH', 'Synthetic Industrial Emulsifier'))
        total_penalty += 4.0
        
    # 6. Palm Oil vs Generic Vegetable Oil
    explicit_palm = ['palm oil', 'palmolein', 'palm fat', 'fractionated fat', 'palm kernel', 'vegetable oil (palm']
    has_explicit_palm = any(k in recipe_text for k in explicit_palm)
    has_generic_oil = ('vegetable oil' in recipe_text or 'edible oil' in recipe_text or 'refined oil' in recipe_text) and not has_explicit_palm
    
    if has_explicit_palm:
        detected_classes.append(('REFINED_PALM_OIL', 3.0, 'FORMULATION_SIGNAL / WATCH', 'Explicit Refined Palm Oil / Palmolein'))
        total_penalty += 3.0
    elif has_generic_oil:
        info_flags.append(('GENERIC_VEGETABLE_OIL_FLAG', 0.0, 'INFORMATIONAL_ONLY', 'Generic Unspecified Vegetable Oil'))
        
    # Apply Component Cap of 25.0 pts
    capped_penalty = min(25.0, total_penalty)
    
    return {
        'detected_classes': detected_classes,
        'info_flags': info_flags,
        'raw_penalty_sum': total_penalty,
        'p_ingredient_capped': capped_penalty
    }

print("✅ Ingredient Policy & Scoring Functions Defined!")


✅ Ingredient Policy & Scoring Functions Defined!


## 🏆 Part 7: Re-Execution on 5 Benchmark Products

We execute the locked formulation policy across our **5 canonical benchmark products**:
1. **Tic Tac** *(Orange Hard Candy)*
2. **Oreo** *(Cadbury Oreo Original Chocolatey Sandwich Biscuits)*
3. **Pringles** *(Potato Chips Desi Masala Tadka)*
4. **7 Up** *(Lemon Soft Drink)*
5. **MAGGI** *(2-Minute Masala Instant Noodles)*


In [3]:
benchmarks_list = [
    'Tic Tac Orange Hard Candy',
    'Cadbury Oreo Original Chocolatey Sandwich\xa0Biscuits',
    'Pringles Potato Chips Desi Masala Tadka Flavour',
    '7 Up Lemon Soft Drink',
    'MAGGI 2-Minute Instant Noodles'
]

bench_results = []
for b_name in benchmarks_list:
    clean_b = b_name.replace('\xa0', ' ').strip()
    match_csv = df_clean[df_clean['Item name'].astype(str).str.replace('\xa0', ' ').str.strip() == clean_b]
    match_llm = [r for r in llm_data if r.get('item_name', '').replace('\xa0', ' ').strip() == clean_b]
    
    ing_text = match_csv.iloc[0]['Ingredients'] if len(match_csv) > 0 else ''
    dec_dict = match_llm[0].get('decoded_e_numbers', {}) if match_llm else {}
    
    res = detect_ingredient_penalties(ing_text, dec_dict)
    
    classes_str = ', '.join([f"{c[0]} (-{c[1]:.0f}pts)" for c in res['detected_classes']]) if res['detected_classes'] else 'None (All Neutral)'
    flags_str = ', '.join([f[0] for f in res['info_flags']]) if res['info_flags'] else 'None'
    
    bench_results.append({
        'Product': clean_b,
        'Detected Ingredient Signals': classes_str,
        'Informational Flags': flags_str,
        'Signal Count': len(res['detected_classes']),
        'P_ingredient (Cap 25)': f"{res['p_ingredient_capped']:.1f} pts",
        'Double-Counting Excluded': 'Sugar, Salt, Starch (Evaluated in P_nutrition)'
    })

df_bench_ing = pd.DataFrame(bench_results)
print("=== 5 BENCHMARK PRODUCTS: INGREDIENT COMPONENT (P_ingredient) LOCKED RESULTS ===")
print(df_bench_ing[['Product', 'Detected Ingredient Signals', 'Informational Flags', 'P_ingredient (Cap 25)']].to_string(index=False))


=== 5 BENCHMARK PRODUCTS: INGREDIENT COMPONENT (P_ingredient) LOCKED RESULTS ===
                                           Product                                                        Detected Ingredient Signals Informational Flags P_ingredient (Cap 25)
                         Tic Tac Orange Hard Candy                                                                 None (All Neutral)                None               0.0 pts
Cadbury Oreo Original Chocolatey Sandwich Biscuits                                                           REFINED_PALM_OIL (-3pts)                None               3.0 pts
   Pringles Potato Chips Desi Masala Tadka Flavour FLAVOR_ENHANCERS (-4pts), INDUSTRIAL_EMULSIFIERS (-4pts), REFINED_PALM_OIL (-3pts)                None              11.0 pts
                             7 Up Lemon Soft Drink                                                     CHEMICAL_PRESERVATIVES (-5pts)                None               5.0 pts
                    MAGGI 2-Minute Inst

## 🧪 Part 8: Policy Validation Suite

We execute 5 controlled validation tests:
1. **Unknown/Unverified Non-Penalization Test**
2. **Double-Counting Exclusion Test**
3. **Monotonicity Under Signal Addition/Removal**
4. **Hard Component Cap Ceiling Test**
5. **Palm Oil vs Generic Vegetable Oil Discrimination Test**


In [4]:
# Run ingredient penalty across all 170 products
all_ing_results = []
for _, row in df_clean.drop_duplicates(subset=['Item name']).iterrows():
    pname = str(row['Item name']).replace('\xa0', ' ').strip()
    match_llm = [r for r in llm_data if r.get('item_name', '').replace('\xa0', ' ').strip() == pname]
    dec_dict = match_llm[0].get('decoded_e_numbers', {}) if match_llm else {}
    
    res = detect_ingredient_penalties(row['Ingredients'], dec_dict)
    all_ing_results.append({
        'Item name': pname,
        'raw_sum': res['raw_penalty_sum'],
        'p_ingredient': res['p_ingredient_capped'],
        'signal_count': len(res['detected_classes'])
    })

df_all_ing = pd.DataFrame(all_ing_results)

# Validation Checks
print("=== INGREDIENT POLICY VALIDATION SUITE ===")
# 1. Unknown non-penalization test
test_unknown = detect_ingredient_penalties("Contains organic ashwagandha extract and natural rosemary aroma (unknown compound xyz)", {})
print(f"1. Unknown Ingredient Safety Test : PASSED (Penalty = {test_unknown['p_ingredient_capped']} pts, 0 false penalty applied).")

# 2. Double-counting check
test_sugar_salt = detect_ingredient_penalties("Contains Cane Sugar, Iodized Salt, Wheat Flour", {})
print(f"2. Double-Counting Exclusion Test: PASSED (Sugar and Salt produce {test_sugar_salt['p_ingredient_capped']} pts in P_ingredient; handled in P_nutrition).")

# 3. Monotonic signal addition
p_base = detect_ingredient_penalties("Refined Wheat Flour, Palm Oil", {})['p_ingredient_capped']
p_added = detect_ingredient_penalties("Refined Wheat Flour, Palm Oil, Preservative (211)", {})['p_ingredient_capped']
print(f"3. Monotonic Signal Addition Test : PASSED (Adding Preservative increased penalty from {p_base} to {p_added} pts).")

# 4. Cap saturation test
extreme_text = "Contains Palm Oil, Preservative (211), Tartrazine, MSG, Sucralose, INS 476"
test_extreme = detect_ingredient_penalties(extreme_text, {'102':'Tartrazine', '211':'Benzoate', '621':'MSG', '955':'Sucralose', '476':'PGPR'})
print(f"4. Hard Component Cap Test        : PASSED (Raw Sum = {test_extreme['raw_penalty_sum']} pts -> Strictly Capped at {test_extreme['p_ingredient_capped']} pts).")

# 5. Generic vegetable oil test
test_generic_oil = detect_ingredient_penalties("Refined Wheat Flour, Edible Vegetable Oil, Salt", {})
print(f"5. Generic Oil Discrimination Test: PASSED (Generic vegetable oil penalty = {test_generic_oil['p_ingredient_capped']} pts; generated flag '{test_generic_oil['info_flags'][0][0]}').")

# 6. Inventory summary statistics
print(f"\n6. Inventory Penalty Spread       : Min = {df_all_ing['p_ingredient'].min()} pts | Median = {df_all_ing['p_ingredient'].median()} pts | Max = {df_all_ing['p_ingredient'].max()} pts")


=== INGREDIENT POLICY VALIDATION SUITE ===
1. Unknown Ingredient Safety Test : PASSED (Penalty = 0.0 pts, 0 false penalty applied).
2. Double-Counting Exclusion Test: PASSED (Sugar and Salt produce 0.0 pts in P_ingredient; handled in P_nutrition).
3. Monotonic Signal Addition Test : PASSED (Adding Preservative increased penalty from 3.0 to 8.0 pts).
4. Hard Component Cap Test        : PASSED (Raw Sum = 30.0 pts -> Strictly Capped at 25.0 pts).
5. Generic Oil Discrimination Test: PASSED (Generic vegetable oil penalty = 0.0 pts; generated flag 'GENERIC_VEGETABLE_OIL_FLAG').

6. Inventory Penalty Spread       : Min = 0.0 pts | Median = 3.0 pts | Max = 15.0 pts


## 📋 Part 9: Final Decision & Architectural Governance Table

| Signal Category | Evidence Classification | Policy Tier | Numeric Deduction | Informational Flag | Policy Rationale |
|---|---|---|---|---|---|
| **Synthetic Colors (Azo & INS 133)** | `FACT FROM SOURCE` | `EVIDENCE_SUPPORTED_SIGNAL` | **$-8.0\text{ pts}$** | `SYNTHETIC_COLOR_FLAG` | Synthetic food colorings subject to cross-border statutory warning labels. |
| **Artificial Sweeteners** | `FACT FROM SOURCE` | `EVIDENCE_SUPPORTED_SIGNAL` | **$-6.0\text{ pts}$** | `ARTIFICIAL_SWEETENER_FLAG` | Non-nutritive high-intensity sweeteners escaping $P_{\text{nutrition}}$ entirely. |
| **Chemical Preservatives** | `FACT FROM SOURCE` | `FORMULATION_SIGNAL / WATCH` | **$-5.0\text{ pts}$** | `PRESERVATIVE_FLAG` | Industrial chemical antimicrobial preservatives (Benzoates, Sorbates). |
| **Flavor Enhancers** | `FACT FROM SOURCE` | `FORMULATION_SIGNAL / WATCH` | **$-4.0\text{ pts}$** | `FLAVOR_ENHANCER_FLAG` | Industrial flavor enhancers (MSG, ribonucleotides). |
| **Industrial Emulsifiers** | `FACT FROM SOURCE` | `FORMULATION_SIGNAL / WATCH` | **$-4.0\text{ pts}$** | `INDUSTRIAL_EMULSIFIER_FLAG` | Synthetic fat replacers / high-intensity emulsifiers (PGPR, DATEM, 471). |
| **Explicit Refined Palm Oil** | `FACT FROM SOURCE` | `FORMULATION_SIGNAL / WATCH` | **$-3.0\text{ pts}$** | `PALM_OIL_FLAG` | Distinct formulation signal evaluated independently of saturated fat. |
| **Generic Vegetable Oil** | `FACT FROM SOURCE` | `UNKNOWN / UNVERIFIED` | **$0.0\text{ pts}$** | `GENERIC_VEGETABLE_OIL_FLAG` | Ambiguous oil identity; no speculative palm oil deduction applied. |
| **Benign Additives (Citric, Lecithin)** | `FACT FROM SOURCE` | `NEUTRAL` | **$0.0\text{ pts}$** | None | Naturally occurring organic acids, plant hydrocolloids, and vitamins. |
| **Unknown / Novel Ingredients** | `FACT FROM SOURCE` | `UNKNOWN / UNVERIFIED` | **$0.0\text{ pts}$** | `UNVERIFIED_INGREDIENT_FLAG` | Unrecognized strings; zero numeric penalty applied. |
| **Hard Component Cap** | `ANTIGRAVITY POLICY DECISION` | **$\text{INGREDIENT\_CAP} = 25.0\text{ pts}$** | Ceiling | None | Prevents additive stacking from dominating overall score. |
| **Double-Counting Exclusions** | `ANTIGRAVITY POLICY DECISION` | **Sugar & Salt Excluded** | $0.0\text{ pts}$ in $P_{\text{ing}}$ | Handled in $P_{\text{nutrition}}$ | Prevents duplicate penalties on identical grams of sucrose / sodium. |

---

### Benchmark Summary for $P_{\text{ingredient}}$:
* **Tic Tac Orange ($0.0\text{ pts}$):** All additives are natural/neutral (Acacia Gum, Citric/Tartaric acid, Carnauba wax).
* **Oreo ($3.0\text{ pts}$):** Contains refined palmolein ($-3.0\text{ pts}$). Emulsifier is benign Soy Lecithin (322).
* **Pringles ($11.0\text{ pts}$):** Contains palm oil ($-3.0$), flavor enhancers INS 627/631 ($-4.0$), and industrial emulsifier INS 471 ($-4.0$).
* **7 Up ($5.0\text{ pts}$):** Contains chemical preservative Sodium Benzoate INS 211 ($-5.0\text{ pts}$).
* **MAGGI ($7.0\text{ pts}$):** Contains palm oil ($-3.0$) and flavor enhancer Disodium Ribonucleotides INS 635 ($-4.0$).

---

$$\mathbf{\text{FINAL STATUS: PHASE\_4C\_READY\_FOR\_COMBINATION}}$$

---

> ### 🛑 HARD BOUNDARY & HARD STOP REACHED
> * **NO** final THRS v3 scores combined yet.
> * **NO** production files modified (`health_scores.json`, `multinational_brand_cleaned.csv`, `recommendations.json`, `app.py`).
> * **NO** JECFA database queried or integrated.
> * **All code, proofs, and tables are self-contained inside [`Phase4C_THRS_v3_Ingredient_Signal_Audit_Design.ipynb`](file:///c:/Users/ARYAN%20PRAJAPATI/OneDrive/Desktop/python/ingredient_platform/Phase4C_THRS_v3_Ingredient_Signal_Audit_Design.ipynb).**
